# Issue #79: OOF誤差分析

`submissions/oof.csv.gz`に保存した時系列CVの検証行だけを分析する。モデルの学習やCVの再計算は行わない。Parquet用ライブラリを追加せずに済むよう、標準的な圧縮CSVを使う。
誤差は`log1p(予測価格) - log1p(正解価格)`で、正なら高く、負なら低く予測したことを表す。

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

# ノートブックをリポジトリ直下・notebooks/ のどちらから開いても読み込める場所を探す。
oof_candidates = [Path("submissions/oof.csv.gz"), Path("../submissions/oof.csv.gz")]
oof_path = next((path for path in oof_candidates if path.is_file()), None)
if oof_path is None:
    raise FileNotFoundError("submissions/oof.csv.gz が見つかりません。先に `uv run python main.py` を実行してください。")
oof = pd.read_csv(oof_path)
prediction_columns = [column for column in oof.columns if column.startswith("予測_")]
model_columns = [column for column in prediction_columns if column not in {"予測_土台", "予測_2段目"}]

oof.shape, sorted(oof["fold"].unique()), prediction_columns


((86274, 30),
 [np.int64(0), np.int64(1), np.int64(2)],
 ['予測_LGBM',
  '予測_LGBM・総額',
  '予測_CatBoost',
  '予測_LinearResidual',
  '予測_LGBM・_built_over_30',
  '予測_LGBM・_area_30_or_less',
  '予測_LGBM・_contract_price',
  '予測_LGBM・_transaction_price',
  '予測_LGBM・_outside_23_wards',
  '予測_Hedonic',
  '予測_TargetEncodedLGBM・基本単位',
  '予測_TargetEncodedLGBM・駅入り',
  '予測_TargetEncodedLGBM・総額・基本単位',
  '予測_土台',
  '予測_2段目'])

## モデルごとの精度

個々のモデル、2段目なしの土台、stacking後の予測を同じOOF行で比較する。値はRMSLEで、小さいほど良い。

In [2]:
def rmsle_by_fold(data: pd.DataFrame, prediction: str) -> pd.DataFrame:
    """指定した予測のRMSLEをfoldごとと全体で計算する。"""
    error = np.log1p(data[prediction]) - np.log1p(data["取引価格（総額）"])
    scores = error.pow(2).groupby(data["fold"]).mean().pow(0.5).rename("RMSLE")
    scores.loc["全体"] = np.sqrt(error.pow(2).mean())
    return scores.to_frame().assign(予測=prediction).reset_index(names="fold")

# 比較対象をこのセル内でも作るため、このセルだけを実行してもNameErrorにならない。
prediction_columns = [column for column in oof.columns if column.startswith("予測_")]
model_scores = pd.concat(
    [rmsle_by_fold(oof, column) for column in prediction_columns],
    ignore_index=True,
)
model_scores.pivot(index="予測", columns="fold", values="RMSLE").sort_values("全体")


fold,0,1,2,全体
予測,,,,
予測_2段目,0.190643,0.181922,0.186097,0.186399
予測_土台,0.193885,0.184220,0.186097,0.188319
予測_LinearResidual,0.195548,0.185876,0.187725,0.189968
予測_TargetEncodedLGBM・基本単位,0.195537,0.185836,0.188256,0.190118
予測_TargetEncodedLGBM・駅入り,0.195568,0.186177,0.188966,0.190461
予測_LGBM,0.196129,0.186294,0.188300,0.190495
予測_CatBoost,0.196205,0.186256,0.188358,0.190530
予測_TargetEncodedLGBM・総額・基本単位,0.196262,0.186875,0.189189,0.191008
予測_LGBM・総額,0.196521,0.186836,0.189209,0.191098


## 区分ごとの誤差と全体への寄与

平均誤差は予測が高めか低めかを示す。「二乗誤差の割合」は、その区分がfold全体の誤差にどれだけ寄与するかを示す。価格帯は検証データの正解価格を使った診断用の区分で、モデルへの入力には使わない。

In [3]:
analysis = oof.copy()
analysis["誤差_log"] = np.log1p(analysis["予測_2段目"]) - np.log1p(analysis["取引価格（総額）"])
analysis["二乗誤差_log"] = analysis["誤差_log"] ** 2
analysis["正解価格帯"] = pd.qcut(analysis["取引価格（総額）"], q=5, duplicates="drop")
analysis["築年数帯"] = pd.cut(analysis["築年数"], bins=[-np.inf, 10, 20, 30, np.inf])
analysis["面積帯"] = pd.cut(analysis["面積_数値"], bins=[0, 30, 50, 70, 100, np.inf], right=False)
analysis["駅距離帯"] = pd.cut(analysis["駅距離_分"], bins=[-np.inf, 5, 10, 20, np.inf])

def summarize_group(data: pd.DataFrame, group: str) -> pd.DataFrame:
    """foldと指定区分ごとに誤差の偏りと二乗誤差の割合を集計する。"""
    result = data.groupby(["fold", group], observed=True).agg(
        件数=("誤差_log", "size"),
        平均誤差_log=("誤差_log", "mean"),
        二乗誤差合計=("二乗誤差_log", "sum"),
    )
    fold_totals = data.groupby("fold")["二乗誤差_log"].sum()
    result["全体の二乗誤差に占める割合"] = result["二乗誤差合計"] / result.index.get_level_values("fold").map(fold_totals)
    result["区分内RMSLE"] = np.sqrt(result["二乗誤差合計"] / result["件数"])
    return result.drop(columns="二乗誤差合計").reset_index()

group_results = {
    column: summarize_group(analysis, column)
    for column in ["正解価格帯", "築年数帯", "面積帯", "駅距離帯", "取引年", "取引四半期", "市区町村名", "間取り", "価格情報区分"]
}
group_results["正解価格帯"]

,fold,正解価格帯,件数,平均誤差_log,全体の二乗誤差に占める割合,区分内RMSLE
0,0,"(99999.999, 24000000.0]",6444,0.087742,0.484535,0.290344
1,0,"(24000000.0, 36000000.0]",5940,0.005012,0.134206,0.159155
2,0,"(36000000.0, 52000000.0]",5759,-0.006509,0.110244,0.146498
3,0,"(52000000.0, 78000000.0]",5814,-0.010727,0.097692,0.137252
4,0,"(78000000.0, 16000000000.0]",6890,-0.026157,0.173323,0.167937
5,1,"(99999.999, 24000000.0]",5923,0.041501,0.475163,0.272977
6,1,"(24000000.0, 36000000.0]",5465,-0.020829,0.144076,0.156486
7,1,"(36000000.0, 52000000.0]",5405,-0.020544,0.110991,0.138109
8,1,"(52000000.0, 78000000.0]",5708,-0.028937,0.103449,0.129747
9,1,"(78000000.0, 16000000000.0]",5565,-0.061931,0.166320,0.166615


## 区分ごとのモデル比較

stacking前の各モデルについても、選んだ区分の中でRMSLEを比べる。あるモデルが一部の区分で相対的に強いかを調べ、stacking後の誤差と合わせて次の候補を考える。

In [4]:
def model_scores_by_group(data: pd.DataFrame, group: str) -> pd.DataFrame:
    """各予測列の区分内RMSLEをfoldごとに計算する。"""
    prediction_columns = [column for column in data.columns if column.startswith("予測_")]
    parts = []
    for prediction in prediction_columns:
        frame = data[["fold", group, "取引価格（総額）", prediction]].copy()
        frame["二乗誤差_log"] = (np.log1p(frame[prediction]) - np.log1p(frame["取引価格（総額）"])) ** 2
        scores = frame.groupby(["fold", group], observed=True)["二乗誤差_log"].mean().pow(0.5)
        parts.append(scores.rename("区分内RMSLE").reset_index().assign(予測=prediction))
    return pd.concat(parts, ignore_index=True)

model_scores_by_group(analysis, "築年数帯")

,fold,築年数帯,区分内RMSLE,予測
0,0,"(-inf, 10.0]",0.151928,予測_LGBM
1,0,"(10.0, 20.0]",0.154337,予測_LGBM
2,0,"(20.0, 30.0]",0.179247,予測_LGBM
3,0,"(30.0, inf]",0.250215,予測_LGBM
4,1,"(-inf, 10.0]",0.147302,予測_LGBM
...,...,...,...,...
175,1,"(30.0, inf]",0.251194,予測_2段目
176,2,"(-inf, 10.0]",0.131660,予測_2段目
177,2,"(10.0, 20.0]",0.153718,予測_2段目
178,2,"(20.0, 30.0]",0.181851,予測_2段目


## 先の四半期数ごとの誤差

検証期間内で何四半期先を予測したかに分ける。先へ行くほど平均誤差が負になるなら、価格を低めに予測する傾向がある。

In [5]:
summarize_group(analysis, "先の四半期数")

,fold,先の四半期数,件数,平均誤差_log,全体の二乗誤差に占める割合,区分内RMSLE
0,0,1,9156,0.017003,0.295928,0.190357
1,0,2,10936,0.004774,0.350117,0.189455
2,0,3,10755,0.009968,0.353954,0.192086
3,1,1,9631,-0.005982,0.331061,0.178687
4,1,2,9660,-0.019102,0.352180,0.184022
5,1,3,8775,-0.028119,0.316759,0.183112
6,2,1,8899,0.012365,0.334630,0.188763
7,2,2,9213,0.015626,0.317366,0.180670
8,2,3,9249,0.013864,0.348004,0.188821


## 大きく外した行

stacking後の絶対誤差が大きい行を見て、複数foldに共通する特徴があるか確認する。個々の価格やIDは分析結果の共有時に必要以上に公開しない。

In [6]:
analysis["絶対誤差_log"] = analysis["誤差_log"].abs()
analysis.sort_values("絶対誤差_log", ascending=False).groupby("fold", observed=True).head(100)[
    ["fold", "id", "取引年", "取引四半期", "取引価格（総額）", "予測_2段目", "誤差_log", "築年数", "面積_数値", "市区町村名", "地区名", "最寄駅：名称", "駅距離_分", "間取り", "価格情報区分"]
]

,fold,id,取引年,取引四半期,取引価格（総額）,予測_2段目,誤差_log,築年数,面積_数値,市区町村名,地区名,最寄駅：名称,駅距離_分,間取り,価格情報区分
75976,2,144646,2023,2,1900000.0,1.881560e+08,4.595417,14.0,130,新宿区,大久保,東新宿,4.0,オープンフロア,不動産取引価格情報
72351,2,114422,2023,4,200000.0,1.476509e+07,4.301699,28.0,45,多摩市,和田,聖蹟桜ケ丘,24.0,２ＤＫ,不動産取引価格情報
54430,1,195391,2024,1,1400000.0,4.865830e+07,3.548349,6.0,30,目黒区,目黒本町,学芸大学,12.0,１ＬＤＫ,不動産取引価格情報
54592,1,196823,2024,1,100000.0,2.343434e+06,3.154193,36.0,15,八王子市,中野山王,八王子,45.0,１Ｋ,不動産取引価格情報
5927,0,43922,2024,4,3600000.0,5.721379e+07,2.765861,29.0,140,大田区,萩中,大鳥居,4.0,NaN,不動産取引価格情報
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56768,1,214329,2024,2,28000000.0,1.147238e+07,-0.892262,32.0,25,立川市,高松町,立川,10.0,１ＬＤＫ,不動産取引価格情報
57313,1,218874,2024,3,67000000.0,2.752377e+07,-0.889643,23.0,25,世田谷区,奥沢,緑が丘(東京),9.0,３ＬＤＫ,不動産取引価格情報
49679,1,155870,2024,3,3500000.0,8.455601e+06,0.882066,31.0,55,青梅市,河辺町,河辺,10.0,３ＬＤＫ,不動産取引価格情報
47831,1,140650,2024,2,7500000.0,1.807900e+07,0.879848,53.0,55,豊島区,南長崎,東長崎,6.0,２ＬＤＫ,成約価格情報


## 次の実験候補

3 foldで同じ向きに誤差が残る区分を候補にする。Issue #72・#74などですでに試した内容、進行中のIssue #81・#83と重ならないか確認してから、候補を一つに絞ってIssueへ記録する。ここで見つけた相関だけで改善を断定せず、次の実験のCVで確かめる。